# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

a30ab34d
How do I submit homework?


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [5]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:3b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [6]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What is the process for turning in assignments?', 'Where should I store my completed work?', "How do I know when it's acceptable to view my submitted answers?", "Can you direct me to the correct folder for each semester's homework?", 'Where are the submission forms located?']


Parallel processing

In [7]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            'question': q,
            'course': doc['course'],
            'document': doc['id']
        })

    return results

Generate questions for all documents:

In [8]:
with ThreadPoolExecutor(max_workers=4) as pool:
    ground_truth = map_progress(pool, documents[:8], process)

  0%|          | 0/8 [00:00<?, ?it/s]

Flatten the nested lists into a single dataset:

In [9]:
ground_truth

[[{'question': 'What is the process for submitting assignments?',
   'course': 'machine-learning-zoomcamp',
   'document': 'a30ab34d'},
  {'question': 'Where can I find the homework files for each cohort?',
   'course': 'machine-learning-zoomcamp',
   'document': 'a30ab34d'},
  {'question': 'How do I access the submission forms on the course platform?',
   'course': 'machine-learning-zoomcamp',
   'document': 'a30ab34d'},
  {'question': 'When will I be able to view my submitted answers?',
   'course': 'machine-learning-zoomcamp',
   'document': 'a30ab34d'},
  {'question': 'What platforms or tools are required to complete and submit homework?',
   'course': 'machine-learning-zoomcamp',
   'document': 'a30ab34d'}],
 [{'question': 'How has the deployment module changed in the 2025 edition?',
   'course': 'machine-learning-zoomcamp',
   'document': '608e975b'},
  {'question': 'What frameworks are now used for neural networks compared to earlier versions?',
   'course': 'machine-learning-zo